# Exploring Filtered Posts

In [9]:
import pandas as pd

In [11]:
filtered = pd.read_csv("data/prelim_filtered_posts_106_11_12_26.csv")

## Choose Sample Post

In [17]:
sample = filtered.sample(5)

In [21]:
example_1 = sample.iloc[0]

In [22]:
example_1

id                                                            1u57s3q
subreddit                                                     resumes
title               [0 YoE, Unemployed, Movie Theatre Floor Staff,...
selftext            Hello, I am a Highschool student applying to t...
url                               https://i.redd.it/n0vogtdnh57h1.png
score                                                              38
num_comments                                                       64
flair                                         Retail/Customer Service
created_utc                                                1781400466
created_datetime                                  2026-06-14 01:27:46
Name: 0, dtype: object

## Extract Relevant Comments

In [26]:
import time
import requests
import pandas as pd

def get_post_comments_arctic(post_id: str, max_retries: int = 3) -> list[dict]:
    """
    Fetches all comments for a single Reddit post using Arctic Shift.
    Handles pagination if the post has more than 100 comments.
    """
    # Clean up ID in case prefix is already present
    clean_id = post_id.replace("t3_", "")
    link_id = f"t3_{clean_id}"
    
    comments = []
    current_before = int(time.time())
    
    while True:
        url = (
            f"https://arctic-shift.photon-reddit.com/api/comments/search"
            f"?link_id={link_id}&limit=100&before={current_before}"
        )
        
        data = None
        for attempt in range(max_retries):
            try:
                res = requests.get(url, timeout=15)
                if res.status_code == 200:
                    data = res.json().get("data", [])
                    break
                elif res.status_code == 429:
                    time.sleep(3)
            except requests.RequestException:
                time.sleep(2)
        
        if not data:
            break
            
        for c in data:
            comments.append({
                "comment_id": c.get("id"),
                "post_id": clean_id,
                "author": c.get("author"),
                "body": c.get("body"),
                "score": c.get("score"),
                "created_utc": c.get("created_utc"),
                "parent_id": c.get("parent_id"),  # Useful to know if it's top-level or reply
            })
            
        # If fewer than 100 comments returned, we've reached the end
        if len(data) < 100:
            break
            
        # Update cursor to get older comments in the next page
        current_before = data[-1]["created_utc"]
        time.sleep(0.5)  # Be polite to the server
        
    return comments

In [27]:
example_1

id                                                            1u57s3q
subreddit                                                     resumes
title               [0 YoE, Unemployed, Movie Theatre Floor Staff,...
selftext            Hello, I am a Highschool student applying to t...
url                               https://i.redd.it/n0vogtdnh57h1.png
score                                                              38
num_comments                                                       64
flair                                         Retail/Customer Service
created_utc                                                1781400466
created_datetime                                  2026-06-14 01:27:46
Name: 0, dtype: object

In [29]:
example_1_comments = get_post_comments_arctic(example_1["id"])

In [30]:
example_1_comments

[{'comment_id': 'ow0xst6',
  'post_id': '1u57s3q',
  'author': 'LeighDax',
  'body': 'Under skills, you never want to put "minor knowledge" as the resume is supposed to highlight you and your experience. If you don\'t feel comfortable enough in the skill, don\'t add it. If you do feel comfortable working with Adobe and photoshop, drop the minor knowledge.',
  'score': 1,
  'created_utc': 1783399706,
  'parent_id': 't3_1u57s3q'},
 {'comment_id': 'ovdw73w',
  'post_id': '1u57s3q',
  'author': 'ctasin_student101',
  'body': 'Hi, just wanted to ask the specific formatting.',
  'score': 1,
  'created_utc': 1783110840,
  'parent_id': 't1_orq8yzx'},
 {'comment_id': 'oscak6j',
  'post_id': '1u57s3q',
  'author': '[deleted]',
  'body': '[removed]',
  'score': 1,
  'created_utc': 1781769844,
  'parent_id': 't3_1u57s3q'},
 {'comment_id': 'ortht3s',
  'post_id': '1u57s3q',
  'author': 'threepointcheese',
  'body': "weird. i live in new york city, and this is how a lot of people get these types of 

In [ ]:
# clean AutoModerator comments and [deleted] or [removed] comments

In [32]:
example_1_comments_cleaned = [comment for comment in example_1_comments if comment["author"] != "AutoModerator" and comment["body"] not in ["[deleted]", "[removed]"]]

In [33]:
example_1_comments_cleaned

[{'comment_id': 'ow0xst6',
  'post_id': '1u57s3q',
  'author': 'LeighDax',
  'body': 'Under skills, you never want to put "minor knowledge" as the resume is supposed to highlight you and your experience. If you don\'t feel comfortable enough in the skill, don\'t add it. If you do feel comfortable working with Adobe and photoshop, drop the minor knowledge.',
  'score': 1,
  'created_utc': 1783399706,
  'parent_id': 't3_1u57s3q'},
 {'comment_id': 'ovdw73w',
  'post_id': '1u57s3q',
  'author': 'ctasin_student101',
  'body': 'Hi, just wanted to ask the specific formatting.',
  'score': 1,
  'created_utc': 1783110840,
  'parent_id': 't1_orq8yzx'},
 {'comment_id': 'ortht3s',
  'post_id': '1u57s3q',
  'author': 'threepointcheese',
  'body': "weird. i live in new york city, and this is how a lot of people get these types of jobs in 2026. im going to go ahead and say you probably don't know what you're talking about or have never really looked for a job like a grown up?",
  'score': 1,
  'creat